# Customer Churn Prediction\n\nThis notebook trains a Machine Learning model (Random Forest Classifier) to predict subscription churn based on customer usage and billing features.\n\n## Objective\n- Identify high-risk customers before they churn.\n- Determine feature importance (which product usage metrics correlate most closely with retention).\n- Export performance metrics for dashboard consumption.

In [ ]:
import pandas as pd\nimport numpy as np\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\n# Set style\nsns.set_theme(style="whitegrid")

## 1. Simulated Data Generation\nFor the purpose of this portfolio demonstration, we simulate realistic SaaS customer telemetry.

In [ ]:
np.random.seed(42)\nn_samples = 5000\n\n# Features\ndata = {\n    'customer_id': range(1, n_samples + 1),\n    'tenure_months': np.random.randint(1, 60, n_samples),\n    'monthly_mrr': np.random.normal(150, 40, n_samples),\n    'engagement_score': np.random.normal(50, 15, n_samples),\n    'support_tickets_30d': np.random.poisson(1.5, n_samples),\n    'last_login_days_ago': np.random.exponential(14, n_samples)\n}\n\ndf = pd.DataFrame(data)\n\n# Target Variable: Churn (Synthetically correlated with features)\n# Higher probability of churn if tenure is low, engagement is low, support tickets are high, and hasn't logged in recently.\nchurn_prob = (\n    (1 - (df['tenure_months'] / 60)) * 0.3 + \n    (1 - (df['engagement_score'] / 100)) * 0.4 + \n    (df['support_tickets_30d'] / 10) * 0.2 + \n    (df['last_login_days_ago'] / 30) * 0.1\n)\n\n# Add some random noise\nchurn_prob += np.random.normal(0, 0.1, n_samples)\ndf['churn'] = (churn_prob > 0.65).astype(int)\n\nprint(df['churn'].value_counts(normalize=True))\ndf.head()

## 2. Model Training

In [ ]:
features = ['tenure_months', 'monthly_mrr', 'engagement_score', 'support_tickets_30d', 'last_login_days_ago']\nX = df[features]\ny = df['churn']\n\nX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)\n\nmodel = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, class_weight='balanced')\nmodel.fit(X_train, y_train)\n\ny_pred = model.predict(X_test)\ny_prob = model.predict_proba(X_test)[:, 1]

## 3. Evaluation Metrics

In [ ]:
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")\nprint(f"Precision: {precision_score(y_test, y_pred):.3f}")\nprint(f"Recall:    {recall_score(y_test, y_pred):.3f}")\nprint(f"F1 Score:  {f1_score(y_test, y_pred):.3f}")\nprint(f"ROC AUC:   {roc_auc_score(y_test, y_prob):.3f}")\n\n# Confusion Matrix\ncm = confusion_matrix(y_test, y_pred)\nplt.figure(figsize=(6, 4))\nsns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)\nplt.title('Confusion Matrix')\nplt.ylabel('True Label')\nplt.xlabel('Predicted Label')\nplt.show()

## 4. Feature Importance

In [ ]:
importances = model.feature_importances_\nindices = np.argsort(importances)[::-1]\n\nplt.figure(figsize=(8, 5))\nsns.barplot(x=importances[indices], y=[features[i] for i in indices], palette="viridis")\nplt.title('Random Forest Feature Importance')\nplt.xlabel('Relative Importance')\nplt.show()